In [ ]:
import sys, warnings, json
from pathlib import Path
import torch
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import get_dataset
from src.features.graph_metrics import compute_graph_metrics
from src.features.preprocessing import preprocess_node_features
from src.models.data import make_node_classification_data, make_link_prediction_data
from src.models import get_model, NodeClassificationTrainer, LinkPredictionTrainer
from src.evaluation.community import run_louvain, evaluate_communities

In [ ]:
ds = get_dataset('cora', verbose=True)
G = ds['graph']
nodes = sorted(G.nodes())
X = preprocess_node_features(ds['features'].reindex(nodes).fillna(0), verbose=False)
y = ds['labels'].reindex(nodes).iloc[:,0].values

In [ ]:
metrics = compute_graph_metrics(G, name='cora',
                                metric_groups=['basic','degree','clustering','paths','community'])
print('=== Cora ===')
print(f'Nodes={G.number_of_nodes()}, Edges={G.number_of_edges()}')
print(f'Density={metrics["density"]:.6f}, AvgDeg={metrics["avg_degree"]:.2f}, '
      f'ClustCoef={metrics["avg_clustering"]:.4f}')

In [ ]:
data_nc = make_node_classification_data(G, y, node_features=X,
                                        test_size=0.2, val_size=0.1, random_state=42)['data']
model_nc = get_model('gcn', num_features=data_nc.x.shape[1],
                     num_classes=len(np.unique(y)), hidden_dim=64, seed=42)
trainer_nc = NodeClassificationTrainer(model_nc, seed=42)
trainer_nc.fit(data_nc, epochs=200, early_stopping=20, verbose=False)
nc_metrics = trainer_nc.evaluate(data_nc, mask=data_nc.test_mask)
torch.save(model_nc.state_dict(), 'models/demo_nc_gcn_cora.pt')
print(f'[Node Classification] GCN Accuracy={nc_metrics["accuracy"]:.4f}, '
      f'MacroF1={nc_metrics["f1_macro"]:.4f}')

In [ ]:
data_lp = make_link_prediction_data(G, node_features=X,
                                    test_ratio=0.1, val_ratio=0.05, random_state=42)['data']
model_lp = get_model('graphsage', num_features=data_lp.x.shape[1],
                     num_classes=2, hidden_dim=64, seed=42)
trainer_lp = LinkPredictionTrainer(model_lp, seed=42)
trainer_lp.fit(data_lp, epochs=80, early_stopping=10, verbose=False)
lp_metrics = trainer_lp.evaluate(data_lp, stage='test')
torch.save(model_lp.state_dict(), 'models/demo_lp_graphsage_cora.pt')
print(f'[Link Prediction] GraphSAGE AUC={lp_metrics["auc"]:.4f}, AP={lp_metrics["ap"]:.4f}')

In [ ]:
partition = run_louvain(G)
true_labels = {n: ds['labels'].iloc[:,0].loc[n] for n in G.nodes()}
cd_metrics = evaluate_communities(true_labels, partition)
print(f'[Community Detection] Louvain NMI={cd_metrics["nmi"]:.4f}, '
      f'ARI={cd_metrics["ari"]:.4f}, Communities={len(set(partition.values()))}')

In [ ]:
results = {
    'dataset': 'cora',
    'node_classification': {
        'model': 'GCN',
        'accuracy': round(nc_metrics['accuracy'], 4),
        'macro_f1': round(nc_metrics['f1_macro'], 4)
    },
    'link_prediction': {
        'model': 'GraphSAGE',
        'auc': round(lp_metrics['auc'], 4),
        'ap': round(lp_metrics['ap'], 4)
    },
    'community_detection': {
        'method': 'Louvain',
        'nmi': round(cd_metrics['nmi'], 4),
        'ari': round(cd_metrics['ari'], 4),
        'n_communities': len(set(partition.values()))
    }
}
Path('results').mkdir(exist_ok=True)
with open('results/demo_cora.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print('\nРезультаты сохранены в results/demo_cora.json')